<a href="https://colab.research.google.com/github/priyakanwarrathod10-art/Face-Recognition-System/blob/main/face_dedation_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install all required libraries
!pip install ultralytics face_recognition opencv-python-headless Pillow -q
!pip install cmake dlib -q  # face_recognition ke liye

In [ ]:
# Cell 2: Imports
import cv2
import numpy as np
import face_recognition
from ultralytics import YOLO
from google.colab import files
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL.Image
import io
import os
import pickle

print("✅ All imports successful!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ All imports successful!


In [ ]:
# Cell 3: Register known people (4 photos upload karo)
# Pehle YOLO model load karo (face detection ke liye)
model = YOLO('yolov8n.pt')  # auto download hoga

known_face_encodings = []
known_face_names = []

def register_person(name):
    """Ek person ki photo upload karke register karo"""
    print(f"\n📸 {name} ki photos upload karo (ek ya zyada):")
    uploaded = files.upload()

    person_encodings = []

    for filename, data in uploaded.items():
        # Image load karo
        img = PIL.Image.open(io.BytesIO(data))
        img_array = np.array(img.convert('RGB'))

        # Face encodings nikalo
        encodings = face_recognition.face_encodings(img_array)

        if encodings:
            person_encodings.append(encodings[0])
            print(f"  ✅ {filename} - Face detected & encoded!")
        else:
            print(f"  ❌ {filename} - Face nahi mila, dobara try karo")

    if person_encodings:
        # Average encoding use karo (multiple photos se better accuracy)
        avg_encoding = np.mean(person_encodings, axis=0)
        known_face_encodings.append(avg_encoding)
        known_face_names.append(name)
        print(f"  🎉 {name} successfully registered!")
    else:
        print(f"  ⚠️ {name} ka koi face encode nahi hua!")

# ========================================
# APNE LOGON KE NAAM YAHAN BADLO
# ========================================
people_to_register = ["Person 1", "Person 2", "Person 3", "Person 4"]

for person_name in people_to_register:
    register_person(person_name)

# Database save karo
with open('known_faces.pkl', 'wb') as f:
    pickle.dump({
        'encodings': known_face_encodings,
        'names': known_face_names
    }, f)

print(f"\n✅ Total {len(known_face_names)} log register hue: {known_face_names}")


📸 Person 1 ki photos upload karo (ek ya zyada):


Saving WhatsApp Image 2026-04-20 at 7.12.34 PM.jpeg to WhatsApp Image 2026-04-20 at 7.12.34 PM.jpeg
  ✅ WhatsApp Image 2026-04-20 at 7.12.34 PM.jpeg - Face detected & encoded!
  🎉 Person 1 successfully registered!

📸 Person 2 ki photos upload karo (ek ya zyada):


Saving WhatsApp Image 2026-04-14 at 2.06.03 PM.jpeg to WhatsApp Image 2026-04-14 at 2.06.03 PM (1).jpeg
  ✅ WhatsApp Image 2026-04-14 at 2.06.03 PM (1).jpeg - Face detected & encoded!
  🎉 Person 2 successfully registered!

📸 Person 3 ki photos upload karo (ek ya zyada):


Saving WhatsApp Image 2026-04-14 at 2.03.51 PM.jpeg to WhatsApp Image 2026-04-14 at 2.03.51 PM (1).jpeg
  ✅ WhatsApp Image 2026-04-14 at 2.03.51 PM (1).jpeg - Face detected & encoded!
  🎉 Person 3 successfully registered!

📸 Person 4 ki photos upload karo (ek ya zyada):


Saving WhatsApp Image 2026-04-14 at 2.07.37 PM.jpeg to WhatsApp Image 2026-04-14 at 2.07.37 PM (1).jpeg
  ✅ WhatsApp Image 2026-04-14 at 2.07.37 PM (1).jpeg - Face detected & encoded!
  🎉 Person 4 successfully registered!

✅ Total 4 log register hue: ['Person 1', 'Person 2', 'Person 3', 'Person 4']


In [ ]:
# Cell 4: Live Camera se face recognition
# (Google Colab mein webcam JavaScript se access hoti hai)

CAPTURE_JS = """
async function captureFrames() {
  const video = document.createElement('video');
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  video.srcObject = stream;
  await video.play();

  // Video element display karo
  document.body.appendChild(video);
  video.style.position = 'fixed';
  video.style.top = '10px';
  video.style.right = '10px';
  video.style.width = '320px';
  video.style.zIndex = '9999';
  video.style.borderRadius = '10px';
  video.style.border = '3px solid #4CAF50';

  const canvas = document.createElement('canvas');

  // 30 frames capture karo
  const frames = [];
  for (let i = 0; i < 30; i++) {
    await new Promise(r => setTimeout(r, 200)); // 200ms gap
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);
    frames.push(canvas.toDataURL('image/jpeg', 0.8));
  }

  stream.getTracks().forEach(t => t.stop());
  video.remove();
  return frames;
}
captureFrames()
"""

def recognize_from_webcam():
    """Webcam se live face recognition"""

    # Known faces load karo
    if os.path.exists('known_faces.pkl'):
        with open('known_faces.pkl', 'rb') as f:
            data = pickle.load(f)
            encodings = data['encodings']
            names = data['names']
    else:
        print("❌ Pehle Cell 3 run karo!")
        return

    print("📷 Camera shuru ho raha hai... (6 seconds capture karega)")

    # Frames capture karo
    js_result = eval_js(CAPTURE_JS)

    print(f"✅ {len(js_result)} frames capture hue!")
    print("\n🔍 Recognition shuru...\n")

    results_log = []

    for i, frame_data in enumerate(js_result):
        # Base64 to image
        img_bytes = b64decode(frame_data.split(',')[1])
        img = PIL.Image.open(io.BytesIO(img_bytes))
        img_rgb = np.array(img.convert('RGB'))
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

        # Face locations detect karo
        face_locations = face_recognition.face_locations(img_rgb)

        if not face_locations:
            continue

        face_encs = face_recognition.face_encodings(img_rgb, face_locations)

        frame_results = []

        for face_enc, (top, right, bottom, left) in zip(face_encs, face_locations):
            # Compare with known faces
            distances = face_recognition.face_distance(encodings, face_enc)

            if len(distances) > 0:
                best_idx = np.argmin(distances)
                best_dist = distances[best_idx]

                THRESHOLD = 0.5  # 0.4 = strict, 0.6 = loose

                if best_dist < THRESHOLD:
                    name = names[best_idx]
                    confidence = round((1 - best_dist) * 100, 1)
                    color = (0, 255, 0)  # Green
                    label = f"✅ {name} ({confidence}%)"
                    message = f"YES! Welcome, {name}! 🎉"
                else:
                    name = "Unknown"
                    color = (0, 0, 255)  # Red
                    label = "❌ Unknown Person"
                    message = "⚠️ Random person wants to enter!"

            # Box draw karo
            cv2.rectangle(img_bgr, (left, top), (right, bottom), color, 2)
            cv2.putText(img_bgr, label, (left, top - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            frame_results.append(message)

        # Har 5th frame show karo
        if i % 5 == 0 and frame_results:
            print(f"Frame {i}: {', '.join(frame_results)}")
            cv2_imshow(img_bgr)

        results_log.extend(frame_results)

    # Final summary
    print("\n" + "="*50)
    print("📊 RECOGNITION SUMMARY")
    print("="*50)

    if results_log:
        from collections import Counter
        counts = Counter(results_log)
        for result, count in counts.most_common():
            print(f"  {result} — {count} frames")
    else:
        print("  Koi face detect nahi hua!")

# Run karo!
recognize_from_webcam()

❌ Pehle Cell 3 run karo!
